(Scalability-ADR)=
# Scalability ADR

We study here the performance of the `ADRSolver` given an increasing number of mesh vertices given volume and surface meshes under different conditions
We will compare:
- `VOL` meshes under Dirichlet and Neumann conditions
- Pure `BND` meshes (no volume elements) closed, open with Dirichlet b.c. and open with Neumann b.c.
- `VOL` meshes with only `BND` species defined and the boundary conditions of the above point

We will time the overall process using the Python module `time` and also the most expensive computation using the Python modules `profile` and `pstats`.

Let's start by performing the analysis for a 2D case


In [ ]:
import cProfile
import pstats
from pstats import SortKey
import time
import numpy as np

from cosmos.utils.generate_surface_meshes import generate_circle, generate_half_sphere, generate_sphere
from ngsolve import *

from cosmos.solvers.simulation import Simulation
from cosmos.solvers.adr import ADRSolver
from cosmos.solvers.tools import gradient

import matplotlib.pyplot as plt

dt = Parameter(0.1)
T = 1
t = Parameter(0)

def volume_problem(dh, bnd):

    t.Set(0)

    mesh, _ = generate_circle(maxh = dh)
    simulation = Simulation(mesh=mesh, t=t, dt = dt, T = T)

    solver = ADRSolver(linear = True)
    simulation.AddSolver(solver)

    u_ex = cos(pi*x)*sin(pi*y)*cos(t) # Exact solution
    d = 1 # diffusion coefficient
    c = 1 # reaction coefficient
    b = CF((2,1)) # advective field
    flux1 = (b*u_ex).Compile() # advective flux
    flux2 = (- d*gradient(u_ex, Id(2))).Compile() # diffusive flux
    flux = flux1 + flux2 # total flux
    rhs = (u_ex.Diff(t) + Trace(gradient(flux, Id(2))) + c*u_ex).Compile() # manufactured solution right-hand side

    neu_b = {'boundary': flux1}
    neu_d = {'boundary': flux2}
    dir_d = {'boundary': u_ex}
    dir_b = {'boundary': u_ex}

    if bnd == 'neu':
        ## Using Neumann boundary conditions
        solver.AddSpecie(VorB = VOL,
                        rhs = rhs,
                        advection = b,
                        diffusion = d,
                        reaction = c,
                        u0 = u_ex,
                        neu_b = neu_b,
                        neu_d = neu_d)
    elif bnd == 'dir':
        solver.AddSpecie(VorB = VOL,
                        rhs = rhs,
                        diffusion = d,
                        reaction = c,
                        advection = b,
                        u0 = u_ex,
                        dir_b = dir_b,
                        dir_d = dir_d)

    start = time.time()
    profiler = cProfile.Profile()
    profiler.enable()
    simulation.run()  # Direct execution
    profiler.disable()
    end = time.time()

    stats = pstats.Stats(profiler)
    stats.strip_dirs().sort_stats(SortKey.TIME).print_stats(3)

    elasped = end-start

    return mesh.nface, elasped

def surface_problem(dh, bnd):

    t.Set(0)

    mesh, _ = generate_half_sphere(maxh = dh)
    simulation = Simulation(mesh=mesh, t=t, dt = dt, T = T)

    solver = ADRSolver(linear = True)
    simulation.AddSolver(solver)

    u_ex = cos(pi*x)*sin(pi*y)*cos(t)
    normal = CF((x,y,z))/Norm(CF((x,y,z)))
    P = Id(3) - OuterProduct(normal, normal)
    d = 1
    c = 1
    b = CF((-z,0,x))
    flux1 = (b*u_ex).Compile()
    flux2 = (- d*gradient(u_ex, P)).Compile()
    flux = flux1 + flux2
    rhs = (u_ex.Diff(t) + Trace(gradient(flux, P)) + c*u_ex).Compile()

    neu_b = {'bottom': flux1}
    neu_d = {'bottom': flux2}
    dir_d = {'bottom': u_ex}
    dir_b = {'bottom': u_ex}

    if bnd == 'neu':
        ## Using Neumann boundary conditions
        solver.AddSpecie(VorB = BND,
                        rhs = rhs,
                        advection = b,
                        diffusion = d,
                        reaction = c,
                        u0 = u_ex,
                        neu_b = neu_b,
                        neu_d = neu_d)
    elif bnd == 'dir':
        solver.AddSpecie(VorB = BND,
                        rhs = rhs,
                        diffusion = d,
                        reaction = c,
                        advection = b,
                        u0 = u_ex,
                        dir_b = dir_b,
                        dir_d = dir_d)

    start = time.time()
    profiler = cProfile.Profile()
    profiler.enable()
    simulation.run()  # Direct execution
    profiler.disable()
    end = time.time()

    stats = pstats.Stats(profiler)
    stats.strip_dirs().sort_stats(SortKey.TIME).print_stats(3)

    elasped = end-start

    return mesh.nface, elasped

def surfacevol_problem(dh, bnd):

    t.Set(0)

    mesh, _ = generate_circle(maxh = dh)
    simulation = Simulation(mesh=mesh, t=t, dt = dt, T = T)

    solver = ADRSolver(linear = True)
    simulation.AddSolver(solver)

    u_ex = cos(pi*x)*sin(pi*y)*cos(t)
    normal = CF((x,y))/Norm(CF((x,y)))
    P = Id(2) - OuterProduct(normal, normal)
    d = 1
    c = 1
    b = CF((-y,x))
    flux1 = (b*u_ex).Compile()
    flux2 = (- d*gradient(u_ex, P)).Compile()
    flux = flux1 + flux2
    rhs = (u_ex.Diff(t) + Trace(gradient(flux, P)) + c*u_ex).Compile()

    ## Using Neumann boundary conditions
    solver.AddSpecie(VorB = BND,
                    rhs = rhs,
                    advection = b,
                    diffusion = d,
                    reaction = c,
                    u0 = u_ex)

    start = time.time()
    profiler = cProfile.Profile()
    profiler.enable()
    simulation.run()  # Direct execution
    profiler.disable()
    end = time.time()

    stats = pstats.Stats(profiler)
    stats.strip_dirs().sort_stats(SortKey.TIME).print_stats(3)

    elasped = end-start

    # 2*mesh.nfacet - 3*mesh.nface

    return mesh.nface, elasped

def get_results(bnd):
    vol_timing = [[], []]
    surf_timing = [[], []]
    surfvol_timing = [[], []]

    max_nv = 1e4
    factor = 1.5
    dh0 = 0.1

    print('='*20)
    print('Volume simulations')
    print('='*20)
    nv = 0
    dh = dh0
    while nv<max_nv:
        nv, elasped = volume_problem(dh, bnd)
        vol_timing[0].append(nv)
        vol_timing[1].append(elasped)
        dh = dh/factor

    print('='*20)
    print('Surface simulations')
    print('='*20)
    nv = 0
    dh = dh0
    while nv<max_nv:
        nv, elasped = surface_problem(dh, bnd)
        surf_timing[0].append(nv)
        surf_timing[1].append(elasped)
        dh = dh/factor

    print('='*20)
    print('SurfacesVolume simulations')
    print('='*20)
    nv = 0
    dh = dh0
    while nv<max_nv:
        nv, elasped = surfacevol_problem(dh, bnd)
        surfvol_timing[0].append(nv)
        surfvol_timing[1].append(elasped)
        dh = dh/factor

    plt.plot(vol_timing[0], vol_timing[1], label = 'vol')
    plt.plot(surf_timing[0], surf_timing[1], label = 'surf')
    plt.plot(surfvol_timing[0], surfvol_timing[1], label = 'surf-vol')

    points = np.concatenate((np.array(vol_timing[0]), np.array(surf_timing[0]), np.array(surfvol_timing[0])))
    points.sort(kind='mergesort')

    # plt.plot(points, points*np.log(points)/2000, label = 'n-log(n)')
    plt.plot(points, points/2000, label = 'n')

    plt.grid()
    plt.xlabel('Number of nodes')
    plt.ylabel('Time [s]')
    plt.legend()
    plt.title(bnd)
    plt.show()

get_results('dir')
get_results('neu')


Volume simulations
         1012 function calls (940 primitive calls) in 0.010 seconds

   Ordered by: internal time
   List reduced from 20 to 3 due to restriction <3>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
     10/1    0.008    0.001    0.001    0.001 utils.py:33(grad)
       10    0.001    0.000    0.001    0.000 tools.py:6(params_update)
        1    0.001    0.001    0.001    0.001 adr.py:71(initialize)


         936 function calls (926 primitive calls) in 0.016 seconds

   Ordered by: internal time
   List reduced from 18 to 3 due to restriction <3>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
     10/1    0.013    0.001    0.001    0.001 utils.py:33(grad)
       10    0.001    0.000    0.002    0.000 tools.py:6(params_update)
        1    0.001    0.001    0.001    0.001 adr.py:71(initialize)


         936 function calls (926 primitive calls) in 0.027 seconds

   Ordered by: internal time
   List reduced from 18 to 

KeyboardInterrupt: 